<a href="https://colab.research.google.com/github/shivani11-glitch/seizure-detection/blob/main/Seizure_processing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# -*- coding: utf-8 -*-
"""
seizure_preprocessing.py
=========================
P1 – Data Preprocessing for Multimodal Seizure Detection
"""

!pip install mne scipy numpy pandas torch openneuro-py tqdm -q

import os, re, gc, glob
import numpy as np
import pandas as pd
import mne
import torch
import openneuro
from scipy.signal import resample, butter, filtfilt
from tqdm import tqdm

BIDS_ROOT    = '/content/seizeit2_data'
TARGET_SFREQ = 256
WINDOW_SEC   = 4
OVERLAP_SEC  = 2
SEMG_CHANNELS = 4

os.makedirs(BIDS_ROOT, exist_ok=True)

openneuro.download(
    dataset='ds005873',
    target_dir=BIDS_ROOT,
    include=[
        'sub-001/ses-01/eeg/*.edf', 'sub-001/ses-01/emg/*.edf', 'sub-001/ses-01/eeg/*.tsv',
        'sub-002/ses-01/eeg/*.edf', 'sub-002/ses-01/emg/*.edf', 'sub-002/ses-01/eeg/*.tsv',
        'sub-003/ses-01/eeg/*.edf', 'sub-003/ses-01/emg/*.edf', 'sub-003/ses-01/eeg/*.tsv',
        'sub-004/ses-01/eeg/*.edf', 'sub-004/ses-01/emg/*.edf', 'sub-004/ses-01/eeg/*.tsv',
        'sub-005/ses-01/eeg/*.edf', 'sub-005/ses-01/emg/*.edf', 'sub-005/ses-01/eeg/*.tsv',
    ]
)

def bandpass_filter(data: np.ndarray, low: float, high: float, fs: float) -> np.ndarray:
    nyq = 0.5 * fs
    high = min(high / nyq, 0.99)
    low  = low / nyq
    b, a = butter(4, [low, high], btype='band')
    return filtfilt(b, a, data)

def get_seizure_events(edf_path: str) -> list:
    parent_dir = os.path.dirname(edf_path).replace('emg', 'eeg')

    # UPDATED: Extract the core BIDS name
    core_id = os.path.basename(edf_path).replace('_eeg.edf', '').replace('_emg.edf', '')

    tsv_files = glob.glob(os.path.join(parent_dir, f'{core_id}*_events.tsv'))
    if not tsv_files:
        tsv_files = glob.glob(os.path.join(parent_dir, '*_events.tsv'))

    if not tsv_files:
        return []

    try:
        df = pd.read_csv(tsv_files[0], sep='\t')
        label_col = next((c for c in df.columns if c.lower() in ['eventtype', 'trial_type', 'label', 'description', 'type']), None)
        if not label_col: return []

        seizures = df[df[label_col].astype(str).str.contains(r'^sz|seiz|ictal|focal', case=False, na=False)]
        return list(zip(seizures['onset'].values, seizures['duration'].values))
    except Exception:
        return []

def extract_semg_channels(raw_emg, n_needed: int = 4) -> np.ndarray:
    data = raw_emg.get_data()
    actual_ch = data.shape[0]
    if actual_ch >= n_needed: return data[:n_needed, :]
    else: return np.concatenate([data] * (n_needed // actual_ch + 1), axis=0)[:n_needed, :]

def load_multimodal_windows(eeg_path: str, emg_path: str, seizure_times: list):
    try:
        raw_eeg = mne.io.read_raw_edf(eeg_path, preload=True, verbose=False)
        raw_emg = mne.io.read_raw_edf(emg_path, preload=True, verbose=False)

        ch_names_upper = [ch.upper() for ch in raw_eeg.ch_names]
        ecg_idx = next((i for i, ch in enumerate(ch_names_upper) if 'ECG' in ch or 'EKG' in ch), 0)

        ecg_raw = bandpass_filter(raw_eeg.get_data()[ecg_idx], 0.5, 45.0, raw_eeg.info['sfreq'])

        emg_raw = extract_semg_channels(raw_emg, n_needed=SEMG_CHANNELS)
        emg_filtered = np.stack([bandpass_filter(emg_raw[i], 20.0, 100.0, raw_emg.info['sfreq']) for i in range(SEMG_CHANNELS)], axis=0)

        n_ecg  = int(len(ecg_raw)      * TARGET_SFREQ / raw_eeg.info['sfreq'])
        n_emg  = int(emg_filtered.shape[1] * TARGET_SFREQ / raw_emg.info['sfreq'])
        common = min(n_ecg, n_emg)

        ecg_res  = resample(ecg_raw, n_ecg)[:common]
        emg_res  = np.stack([resample(emg_filtered[i], n_emg)[:common] for i in range(SEMG_CHANNELS)], axis=0)

        ecg_res = (ecg_res - ecg_res.mean()) / (ecg_res.std() + 1e-8)
        for i in range(SEMG_CHANNELS):
            emg_res[i] = (emg_res[i] - emg_res[i].mean()) / (emg_res[i].std() + 1e-8)

        win_len = int(WINDOW_SEC * TARGET_SFREQ)
        step    = int((WINDOW_SEC - OVERLAP_SEC) * TARGET_SFREQ)

        X_ecg, X_semg, y = [], [], []

        for s in range(0, common - win_len, step):
            e   = s + win_len
            t_s = s / TARGET_SFREQ
            t_e = e / TARGET_SFREQ
            label = 1 if any(t_s < (on + dur) and t_e > on for (on, dur) in seizure_times) else 0

            X_ecg.append(ecg_res[s:e][np.newaxis, :])
            X_semg.append(emg_res[:, s:e])
            y.append(label)

        raw_eeg.close()
        raw_emg.close()
        gc.collect()

        return np.array(X_ecg, dtype=np.float32), np.array(X_semg, dtype=np.float32), np.array(y, dtype=np.int64)
    except Exception:
        return None, None, None

all_pairs = {}
for f in sorted(glob.glob(os.path.join(BIDS_ROOT, '**', '*.edf'), recursive=True)):
    # UPDATED: Group files by their actual base name
    file_name = os.path.basename(f)
    core_id = file_name.replace('_eeg.edf', '').replace('_emg.edf', '')

    all_pairs.setdefault(core_id, {})
    if '_eeg.edf' in f: all_pairs[core_id]['eeg'] = f
    if '_emg.edf' in f: all_pairs[core_id]['emg'] = f

all_ecg, all_semg, all_y = [], [], []

# UPDATED: Slicing to 3 patients to prevent RAM crash during P3 integration test
test_pairs = dict(list(all_pairs.items())[:3])

for run_id, paths in tqdm(test_pairs.items(), desc="Processing runs"):
    if 'eeg' not in paths or 'emg' not in paths: continue
    sz_times = get_seizure_events(paths['eeg'])
    X_ecg, X_semg, y  = load_multimodal_windows(paths['eeg'], paths['emg'], sz_times)
    if X_ecg is not None and len(X_ecg) > 0:
        all_ecg.append(X_ecg)
        all_semg.append(X_semg)
        all_y.append(y)

if all_ecg:
    ECG_final  = np.concatenate(all_ecg,  axis=0)
    sEMG_final = np.concatenate(all_semg, axis=0)
    y_final    = np.concatenate(all_y,    axis=0)
    print(f"\n✅ Preprocessing complete! (Tested on {len(test_pairs)} patients)")
else:
    print("\n❌ No data loaded.")